In [1]:
import os, random, time, json, math, copy
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

WORK_DIR = "/kaggle/working"
CKPT_DIR = os.path.join(WORK_DIR, "checkpoints")
RESULTS_DIR = os.path.join(WORK_DIR, "results")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Ready.")

Device: cuda | GPU: Tesla T4
Ready.


In [2]:
!pip install -q timm==1.0.9 torchmetrics==1.4.1 opencv-python-headless scikit-learn albumentations requests openpyxl thop --no-warn-script-location
print("Libraries installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.2/866.2 kB 31.4 MB/s eta 0:00:00
Libraries installed.


In [3]:
import cv2
import requests
import pandas as pd
from collections import Counter

# ---- Correct paths based on actual structure (this notebook) ----
PCOS_XAI_DIR         = "/kaggle/input/datasets/ibadeus/pcos-xai-ultrasound-dataset"
PCOSGEN_TRAIN_IMAGES = "/kaggle/input/datasets/arunthathid/gentrain/PCOSGen-train/images"
PCOSGEN_TRAIN_LABELS = "/kaggle/input/datasets/arunthathid/gentrain/PCOSGen-train/class_label.xlsx"
PCOSGEN_TEST_IMAGES  = "/kaggle/input/datasets/arunthathid/gentest/images"
TEST_LABELS_PATH     = "/kaggle/working/test_class_label.csv"

# ---- PCOS-XAI: folder-based labels ----
def list_from_folders(root, class_map):
    samples = []
    for dirpath, _, files in os.walk(root):
        label = class_map.get(os.path.basename(dirpath), None)
        if label is None:
            continue
        for fname in files:
            if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                samples.append((os.path.join(dirpath, fname), label))
    return samples

xai_samples = list_from_folders(PCOS_XAI_DIR, {"infected": 1, "noninfected": 0})
print(f"PCOS-XAI raw: {len(xai_samples)}")
def load_from_label_file(label_path, images_dir):
    df = pd.read_csv(label_path) if label_path.endswith(".csv") else pd.read_excel(label_path)
    samples, missing = [], 0
    for _, row in df.iterrows():
        img_path = os.path.join(images_dir, row["imagePath"])
        if os.path.exists(img_path):
            samples.append((img_path, 0 if row["Healthy"] == 1 else 1))
        else:
            missing += 1
    print(f"  {os.path.basename(label_path)}: {len(samples)} loaded, {missing} missing")
    return samples

# fetch test labels CSV if not already present
if not os.path.exists(TEST_LABELS_PATH):
    print("Fetching PCOSGen test labels from Zenodo...")
    r = requests.get("https://zenodo.org/records/10960327/files/class%20label.csv?download=1", timeout=60)
    r.raise_for_status()
    with open(TEST_LABELS_PATH, "wb") as f:
        f.write(r.content)
    print("Done.")

pcosgen_train   = load_from_label_file(PCOSGEN_TRAIN_LABELS, PCOSGEN_TRAIN_IMAGES)
pcosgen_test    = load_from_label_file(TEST_LABELS_PATH, PCOSGEN_TEST_IMAGES)
pcosgen_samples = pcosgen_train + pcosgen_test

print(f"PCOSGen raw: {len(pcosgen_samples)}")
print(f"Total raw: {len(xai_samples) + len(pcosgen_samples)}")
print("PCOS-XAI distribution:", Counter([s[1] for s in xai_samples]))
print("PCOSGen distribution:", Counter([s[1] for s in pcosgen_samples]))

PCOS-XAI raw: 11784
Fetching PCOSGen test labels from Zenodo...
Done.
  class_label.xlsx: 3200 loaded, 0 missing
  test_class_label.csv: 1468 loaded, 0 missing
PCOSGen raw: 4668
Total raw: 16452
PCOS-XAI distribution: Counter({1: 6784, 0: 5000})
PCOSGen distribution: Counter({1: 3348, 0: 1320})


In [4]:
import hashlib

def md5_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

print("Deduplicating PCOS-XAI...")
seen_md5 = set()
dedup_xai = []
for path, label in xai_samples:
    h = md5_hash(path)
    if h not in seen_md5:
        seen_md5.add(h)
        dedup_xai.append((path, label))
print(f"  PCOS-XAI: {len(xai_samples)} -> {len(dedup_xai)} unique")

print("Deduplicating PCOSGen (also removing cross-dataset duplicates)...")
dedup_pcosgen = []
for path, label in pcosgen_samples:
    h = md5_hash(path)
    if h not in seen_md5:
        seen_md5.add(h)
        dedup_pcosgen.append((path, label))
print(f"  PCOSGen: {len(pcosgen_samples)} -> {len(dedup_pcosgen)} unique")

dedup_xai_tagged     = [(p, l, 0) for p, l in dedup_xai]
dedup_pcosgen_tagged = [(p, l, 1) for p, l in dedup_pcosgen]
all_samples = dedup_xai_tagged + dedup_pcosgen_tagged

print(f"\nCombined unique samples: {len(all_samples)}")
print("Class distribution:", Counter([s[1] for s in all_samples]))
print("Source distribution (0=XAI, 1=PCOSGen):", Counter([s[2] for s in all_samples]))

Deduplicating PCOS-XAI...
  PCOS-XAI: 11784 -> 3996 unique
Deduplicating PCOSGen (also removing cross-dataset duplicates)...
  PCOSGen: 4668 -> 2295 unique

Combined unique samples: 6291
Class distribution: Counter({1: 4202, 0: 2089})
Source distribution (0=XAI, 1=PCOSGen): Counter({0: 3996, 1: 2295})


In [5]:
def apply_clahe(img, clip_limit=2.0, tile_grid_size=(8, 8)):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    enhanced = clahe.apply(gray)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB)

def load_and_preprocess(path, img_size=224):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    img = apply_clahe(img)
    return cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)

test_img = load_and_preprocess(all_samples[0][0])
print("Sample shape:", test_img.shape, "dtype:", test_img.dtype)

Sample shape: (224, 224, 3) dtype: uint8


In [6]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.GaussNoise(std_range=(0.02, 0.08), p=0.2),
    A.ElasticTransform(alpha=1, sigma=20, p=0.15),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])
eval_transform = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])
print("Augmentation pipelines ready.")

Augmentation pipelines ready.


In [7]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class PCOSDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label, source = self.samples[idx]
        img = load_and_preprocess(path)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, torch.tensor(label, dtype=torch.long), torch.tensor(source, dtype=torch.long)

labels_all = [s[1] for s in all_samples]
train_val, test = train_test_split(all_samples, test_size=0.2, stratify=labels_all, random_state=SEED)
train, val = train_test_split(train_val, test_size=0.125,
                               stratify=[s[1] for s in train_val], random_state=SEED)

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

BATCH_SIZE = 16
train_loader = DataLoader(PCOSDataset(train, train_transform), batch_size=BATCH_SIZE,
                           shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(PCOSDataset(val, eval_transform), batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(PCOSDataset(test, eval_transform), batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=2, pin_memory=True)

train_labels = [s[1] for s in train]
class_counts = Counter(train_labels)
w_neg = len(train) / (2 * class_counts[0])
w_pos = len(train) / (2 * class_counts[1])
class_weights = torch.tensor([w_neg, w_pos], dtype=torch.float).to(DEVICE)
ce_loss_fn = nn.CrossEntropyLoss(weight=class_weights)
print(f"Class weights: negative={w_neg:.3f}, positive={w_pos:.3f}")

Train: 4403 | Val: 629 | Test: 1259
Class weights: negative=1.506, positive=0.749


In [8]:
import timm
import torch.optim as optim
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, matthews_corrcoef, confusion_matrix,
                              roc_curve, auc, precision_recall_curve, average_precision_score)

def train_and_test_baseline(bmodel, model_name, num_epochs=40, lr=1e-4, patience=8):
    optimizer = optim.Adam(bmodel.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=4, factor=0.5)
    best_val_acc = 0.0
    best_state = None
    patience_counter = 0
    ckpt_path = os.path.join(CKPT_DIR, f"baseline_{model_name}.pt")

    for epoch in range(1, num_epochs + 1):
        bmodel.train()
        run_loss, correct, total = 0.0, 0, 0
        for imgs, labels, sources in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = bmodel(imgs)
            loss = ce_loss_fn(logits, labels)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * imgs.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
        train_acc = correct / total

        bmodel.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, labels, sources in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                preds = bmodel(imgs).argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += imgs.size(0)
        val_acc = val_correct / val_total
        scheduler.step(val_acc)
        print(f"[{model_name}] Epoch {epoch:02d}/{num_epochs} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            best_state = {k: v.cpu() for k, v in bmodel.state_dict().items()}
            torch.save({"epoch": epoch, "val_acc": best_val_acc, "model_state_dict": best_state}, ckpt_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    print(f"\n{model_name} — Best val acc: {best_val_acc:.4f}")
    bmodel.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    bmodel.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels, sources in test_loader:
            imgs = imgs.to(DEVICE)
            logits = bmodel(imgs)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(dim=1)
            all_labels.extend(labels.numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    yt, yp, yprob = np.array(all_labels), np.array(all_preds), np.array(all_probs)
    print(f"\n=== {model_name} Test Metrics ===")
    print(f"Accuracy:  {accuracy_score(yt,yp)*100:.2f}%  Precision: {precision_score(yt,yp)*100:.2f}%  "
          f"Recall: {recall_score(yt,yp)*100:.2f}%")
    print(f"F1 Score:  {f1_score(yt,yp)*100:.2f}%  MCC: {matthews_corrcoef(yt,yp)*100:.2f}%")
    return yt, yp, yprob

print("train_and_test_baseline() ready.")

train_and_test_baseline() ready.


In [9]:
class RepViTM2(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model("repvit_m2", pretrained=pretrained, num_classes=0, global_pool="avg")
        feat_ch = self.backbone.num_features
        self.classifier = nn.Sequential(nn.Linear(feat_ch, feat_ch//2), nn.ReLU(inplace=True),
                                         nn.Dropout(0.3), nn.Linear(feat_ch//2, num_classes))
    def forward(self, x):
        return self.classifier(self.backbone(x))

model_repvit = RepViTM2(pretrained=True).to(DEVICE)
print(f"RepViT-M2 params: {sum(p.numel() for p in model_repvit.parameters())/1e6:.2f}M")
y_true_repvit, y_pred_repvit, y_prob_repvit = train_and_test_baseline(model_repvit, "repvit_m2")

model.safetensors:   0%|          | 0.00/35.5M [00:00<?, ?B/s]

Unexpected keys (head.head.bn.num_batches_tracked, head.head.bn.bias, head.head.bn.running_mean, head.head.bn.running_var, head.head.bn.weight, head.head_dist.bn.num_batches_tracked, head.head_dist.bn.bias, head.head_dist.bn.running_mean, head.head_dist.bn.running_var, head.head_dist.bn.weight) found while loading pretrained weights. This may be expected if model is being adapted.


RepViT-M2 params: 7.90M
[repvit_m2] Epoch 01/40 | train_acc=0.7472 | val_acc=0.7488
[repvit_m2] Epoch 02/40 | train_acc=0.8013 | val_acc=0.8188
[repvit_m2] Epoch 03/40 | train_acc=0.8163 | val_acc=0.8362
[repvit_m2] Epoch 04/40 | train_acc=0.8256 | val_acc=0.8076
[repvit_m2] Epoch 05/40 | train_acc=0.8335 | val_acc=0.7981
[repvit_m2] Epoch 06/40 | train_acc=0.8313 | val_acc=0.8235
[repvit_m2] Epoch 07/40 | train_acc=0.8381 | val_acc=0.8092
[repvit_m2] Epoch 08/40 | train_acc=0.8478 | val_acc=0.7981
[repvit_m2] Epoch 09/40 | train_acc=0.8510 | val_acc=0.8108
[repvit_m2] Epoch 10/40 | train_acc=0.8617 | val_acc=0.8156
[repvit_m2] Epoch 11/40 | train_acc=0.8674 | val_acc=0.7949
Early stopping at epoch 11

repvit_m2 — Best val acc: 0.8362

=== repvit_m2 Test Metrics ===
Accuracy:  84.75%  Precision: 87.09%  Recall: 90.61%
F1 Score:  88.81%  MCC: 65.03%


In [10]:
# ==============================================================
# REPVIT-M2 — ROC-AUC AND AVERAGE PRECISION
# ==============================================================

from sklearn.metrics import roc_auc_score, average_precision_score

# ROC-AUC
roc_auc_repvit = roc_auc_score(
    y_true_repvit,
    y_prob_repvit
)

# Average Precision
ap_repvit = average_precision_score(
    y_true_repvit,
    y_prob_repvit
)

print("=" * 60)
print("RepViT-M2 — ROC-AUC RESULTS")
print("=" * 60)
print(f"Test samples       : {len(y_true_repvit)}")
print(f"ROC-AUC             : {roc_auc_repvit:.4f}")
print(f"Average Precision   : {ap_repvit:.4f}")
print("=" * 60)

RepViT-M2 — ROC-AUC RESULTS
Test samples       : 1259
ROC-AUC             : 0.9388
Average Precision   : 0.9705


In [11]:
RESULT_KEY = "repvit_m2"
np.savez(os.path.join(RESULTS_DIR, f"{RESULT_KEY}.npz"),
         y_true=y_true_repvit, y_pred=y_pred_repvit, y_prob=y_prob_repvit)
print(f"Saved results for {RESULT_KEY} -> {RESULTS_DIR}/{RESULT_KEY}.npz")

Saved results for repvit_m2 -> /kaggle/working/results/repvit_m2.npz
